# Lab — DQN with Double Learning
**AI Agents · UC2 DQN · 2.3** — Student lab guide

Complete every cell marked **TODO** (replace `raise NotImplementedError()` with your code) and answer every **✍️ question**.

## Learning objectives

By the end of this lab you will be able to:

1. Implement the building blocks of DQN in PyTorch: a Q-network, an experience-replay buffer and an ε-greedy policy.
2. Implement **both** TD targets — standard DQN and **Double DQN** — and explain the single line that differs.
3. Handle terminal states correctly (and tell a *terminal* state apart from a *time-limit truncation*).
4. Train both agents on `CartPole-v1` and **measure overestimation** of action values.
5. Study the effect of the target-network update interval **C**.

**Reference:** M. Hu, *The Art of Reinforcement Learning: Fundamentals, Mathematics, and Implementations with Python*, Apress, 2023 — Section 2.3, *DQN with Double Learning*.

**Estimated time:** 2–3 hours (training runs take a few minutes each on a laptop CPU).

## The idea in one sentence

> **One network chooses the action, the other one judges it.**

| | TD target $\delta$ |
|---|---|
| Standard DQN | $r + \gamma \, \max_{a'} \hat Q(s', a'; \theta^-)$ |
| **Double DQN** | $r + \gamma \, \hat Q\big(s', \underbrace{\arg\max_{a'} \hat Q(s', a'; \theta)}_{\theta \text{ selects}};\ \underbrace{\theta^-}_{\theta^- \text{ evaluates}}\big)$ |

and the objective is $J(\theta) = \big(\delta - \hat Q(s, a; \theta)\big)^2$, with $\delta$ treated as a **constant** (no gradient).

## How this notebook maps to the algorithm

| Notebook part | Algorithm line(s) | What you implement |
|---|---|---|
| Part 2 — Q-network | Initialize θ, θ⁻ | `QNetwork` |
| Part 3 — Replay buffer | 6, 9 | `ReplayBuffer.push`, `ReplayBuffer.sample` |
| Part 4 — ε-greedy | 2, 5 | `linear_epsilon`, `select_action` |
| Part 5 — TD targets | **10** | `dqn_target`, `double_dqn_target` |
| Part 6 — Loss and SGD | 11, 12 | `optimize_step` |
| Part 7 — Training loop | 1–13 | storing transitions, warm-up gate, target sync |
| Parts 8–9 — Experiments | — | compare DQN vs Double DQN, vary C |

Every **TODO** is followed by a **check cell**. Run it: if it prints ✅ you can move on.

---
## Part 0 — Setup

If you are on Google Colab, PyTorch and Gymnasium are usually already installed. Otherwise, uncomment the first line.

In [ ]:
# %pip install -q gymnasium torch matplotlib numpy

import os

os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| Gymnasium", gym.__version__, "| device:", device)


def set_seed(seed: int) -> None:
    """Make runs reproducible (as far as PyTorch allows)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(0)

---
## Part 1 — Meet the environment: `CartPole-v1`

A pole is attached to a cart. The agent pushes the cart **left (0)** or **right (1)**. It receives **+1 for every step** the pole stays up.

* **State** (4 numbers): cart position, cart velocity, pole angle, pole angular velocity.
* **Terminated**: the pole falls past ±12° or the cart leaves the track → a real terminal state.
* **Truncated**: the episode reaches 500 steps → the time limit stops it, but the state itself is **not** terminal.

Keep that last distinction in mind; it matters in Part 5.

With $\gamma = 0.99$ and a reward of +1 per step, no state can be worth more than $\sum_t \gamma^t = \frac{1}{1-\gamma} = 100$. Any Q-value above 100 is therefore an overestimate. We will use this as a free "ground truth" in Part 8.

### TODO 1.1 — Baseline: a random policy

Run **20 episodes** with uniformly random actions and store the undiscounted return of each one in `random_returns`. This gives the baseline every learning agent must beat.

In [ ]:
env = gym.make("CartPole-v1")
print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)

random_returns = []
# YOUR CODE HERE
raise NotImplementedError()
env.close()

print(f"Random policy: mean return = {np.mean(random_returns):.1f} ± {np.std(random_returns):.1f}")

In [ ]:
# ✔ Check 1.1
assert len(random_returns) == 20, "Store one return per episode (20 in total)."
assert 8 <= np.mean(random_returns) <= 60, "A random policy usually scores around 20 on CartPole."
print("✅ Part 1 OK")

---
## Part 2 — The Q-network $\hat Q(s, \cdot\,; \theta)$

The network takes a state and outputs **one Q-value per action**. A single forward pass therefore scores every action at once, which is what makes $\max$ and $\arg\max$ cheap.

### TODO 2.1 — Implement `QNetwork`

An MLP: `obs_dim → hidden → hidden → n_actions`, with ReLU between layers and **no activation on the output** (Q-values can be any real number).

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, obs_dim) -> Q-values: (batch, n_actions)"""
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
# ✔ Check 2.1
_net = QNetwork(4, 2)
_out = _net(torch.randn(5, 4))
assert _out.shape == (5, 2), f"Expected shape (5, 2), got {tuple(_out.shape)}"
assert (_out < 0).any() or (_out > 0).any(), "Outputs look constant; check your layers."
print("✅ Part 2 OK —", sum(p.numel() for p in _net.parameters()), "parameters")

---
## Part 3 — Experience replay (algorithm lines 6 and 9)

Consecutive transitions are strongly correlated (frame *t* looks almost like frame *t+1*). Storing them in a buffer $D$ and sampling **random** mini-batches breaks that correlation and lets each transition be reused many times.

### TODO 3.1 — `push` and `sample`

* `push` stores one transition `(s, a, r, s_next, done)`. The `deque(maxlen=capacity)` drops the oldest one automatically.
* `sample` draws `batch_size` transitions **uniformly at random, without replacement**, and returns five tensors on `device`:

| tensor | shape | dtype |
|---|---|---|
| states, next_states | `(b, obs_dim)` | `float32` |
| actions | `(b,)` | `int64` (needed later by `gather`) |
| rewards, dones | `(b,)` | `float32` |

*Tip:* `zip(*batch)` turns a list of tuples into a tuple of lists. Convert lists of arrays with `np.array(...)` before `torch.as_tensor` (it's much faster).

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done) -> None:
        # YOUR CODE HERE
        raise NotImplementedError()

    def sample(self, batch_size: int):
        # YOUR CODE HERE
        raise NotImplementedError()

    def __len__(self) -> int:
        return len(self.buffer)

In [ ]:
# ✔ Check 3.1
_buf = ReplayBuffer(capacity=50)
for i in range(80):
    _buf.push(np.full(4, i, dtype=np.float32), i % 2, 1.0, np.full(4, i + 1, dtype=np.float32), float(i % 10 == 0))
assert len(_buf) == 50, "The buffer must never grow past its capacity."
_s, _a, _r, _s2, _d = _buf.sample(16)
assert _s.shape == (16, 4) and _s2.shape == (16, 4), "States must have shape (batch, obs_dim)."
assert _a.shape == (16,) and _a.dtype == torch.int64, "Actions must be a 1-D int64 tensor."
assert _r.dtype == torch.float32 and _d.dtype == torch.float32, "Rewards and dones must be float32."
assert _s.min().item() >= 30, "Old transitions should have been dropped (FIFO)."
assert torch.allclose(_s2, _s + 1), "next_state must stay aligned with its state."
print("✅ Part 3 OK")

---
## Part 4 — ε-greedy exploration (algorithm lines 2 and 5)

* **Line 2:** with probability ε take a random action, otherwise take $\arg\max_a \hat Q(s, a; \theta)$. Note that it's the **online** network θ that acts, never θ⁻.
* **Line 5:** decay ε linearly from `eps_start` to `eps_end` over `decay_steps`, then keep it fixed.

### TODO 4.1 — `linear_epsilon` and `select_action`

Wrap the forward pass in `torch.no_grad()`: choosing an action is not training.

In [ ]:
def linear_epsilon(step: int, eps_start: float, eps_end: float, decay_steps: int) -> float:
    # YOUR CODE HERE
    raise NotImplementedError()


def select_action(state: np.ndarray, q_net: nn.Module, epsilon: float, n_actions: int) -> int:
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# ✔ Check 4.1
assert abs(linear_epsilon(0, 1.0, 0.05, 1000) - 1.0) < 1e-9
assert abs(linear_epsilon(500, 1.0, 0.05, 1000) - 0.525) < 1e-9
assert abs(linear_epsilon(1000, 1.0, 0.05, 1000) - 0.05) < 1e-9
assert abs(linear_epsilon(5000, 1.0, 0.05, 1000) - 0.05) < 1e-9, "ε must stop decaying at eps_end."


class _Fixed(nn.Module):  # always prefers action 1
    def forward(self, x):
        return torch.tensor([[0.0, 1.0]], device=x.device).repeat(x.shape[0], 1)


_greedy = [select_action(np.zeros(4), _Fixed(), 0.0, 2) for _ in range(50)]
assert all(a == 1 for a in _greedy), "With ε = 0 the agent must always take the argmax."
_rand = [select_action(np.zeros(4), _Fixed(), 1.0, 2) for _ in range(2000)]
assert 0.4 < np.mean(_rand) < 0.6, "With ε = 1 actions must be uniformly random."
print("✅ Part 4 OK")

---
## Part 5 — The TD targets (algorithm line 10) ⭐ the heart of the lab

$$
\delta_j =
\begin{cases}
r_j & \text{if } s_{j+1} \text{ is terminal}\\[4pt]
r_j + \gamma\, \hat Q\big(s_{j+1}, \arg\max_{a'} \hat Q(s_{j+1}, a'; \theta);\ \theta^-\big) & \text{otherwise (bootstrapping)}
\end{cases}
$$

> ⚠️ The algorithm box in the reference writes $r_t$ in the bootstrapping branch. Read it as $r_j$, the reward of the **sampled** transition.

Both cases fit in one vectorized line by multiplying the bootstrap term by `(1 - done)`.

**Useful PyTorch tools:**
* `q.max(dim=1).values` → the max over actions, shape `(b,)`
* `q.argmax(dim=1, keepdim=True)` → the index of the best action, shape `(b, 1)`
* `q.gather(1, idx)` → picks `q[i, idx[i]]` for each row, shape `(b, 1)`; then `.squeeze(1)`

### TODO 5.1 — Standard DQN target
θ⁻ both **selects** and **evaluates**: $r + \gamma(1-d)\max_{a'} \hat Q(s', a'; \theta^-)$.

### TODO 5.2 — Double DQN target
θ **selects** (`argmax` on `online_net`), θ⁻ **evaluates** (`gather` on `target_net`).

Both functions run under `torch.no_grad()`: the target is a **label**, not something we differentiate.

In [ ]:
def dqn_target(rewards, next_states, dones, online_net, target_net, gamma):
    """Standard DQN: max over θ⁻. (online_net is unused; kept for a common signature.)"""
    with torch.no_grad():
        # YOUR CODE HERE
        raise NotImplementedError()


def double_dqn_target(rewards, next_states, dones, online_net, target_net, gamma):
    """Double DQN: θ selects the next action, θ⁻ evaluates it."""
    with torch.no_grad():
        # YOUR CODE HERE
        raise NotImplementedError()

### ✔ Check 5 — the worked example from the slides

Next state $s'$ has three actions, with $r = 1$ and $\gamma = 0.9$:

| action | online θ | target θ⁻ |
|---|---|---|
| a₁ | 2.1 | 2.5 |
| a₂ | **3.4** ← θ's argmax | 2.9 ← Double DQN reads this |
| a₃ | 2.8 | 3.6 ← standard DQN's max |

Expected: standard DQN → $1 + 0.9 \times 3.6 = 4.24$; Double DQN → $1 + 0.9 \times 2.9 = 3.61$.
The second row of the batch is the same transition, but **terminal**, so both targets must equal $r = 1$.

In [ ]:
# ✔ Check 5
class _ConstQ(nn.Module):
    """A fake network that returns fixed Q-values, whatever the input."""
    def __init__(self, values):
        super().__init__()
        self.values = torch.tensor(values, dtype=torch.float32, device=device)

    def forward(self, x):
        return self.values


_online = _ConstQ([[2.1, 3.4, 2.8], [2.1, 3.4, 2.8]])
_target = _ConstQ([[2.5, 2.9, 3.6], [2.5, 2.9, 3.6]])
_r = torch.tensor([1.0, 1.0], device=device)
_d = torch.tensor([0.0, 1.0], device=device)          # row 1 is terminal
_s2 = torch.zeros(2, 4, device=device)

_std = dqn_target(_r, _s2, _d, _online, _target, 0.9)
_dbl = double_dqn_target(_r, _s2, _d, _online, _target, 0.9)
print("Standard DQN targets:", _std.tolist())
print("Double DQN targets:  ", _dbl.tolist())

assert _std.shape == (2,) and _dbl.shape == (2,), "Targets must have shape (batch,). Did you forget .squeeze(1)?"
assert torch.allclose(_std, torch.tensor([4.24, 1.0], device=device), atol=1e-5), "Standard DQN target is wrong."
assert torch.allclose(_dbl, torch.tensor([3.61, 1.0], device=device), atol=1e-5), \
    "Double DQN target is wrong. Who selects (argmax), who evaluates (gather)?"
assert not _dbl.requires_grad, "Targets must not carry gradients."
print("✅ Part 5 OK — you just implemented the only line that makes DQN 'Double'.")

### ✍️ Question 5 (answer in a sentence or two)

**(a)** In CartPole, when an episode ends at step 500 because of the time limit (`truncated=True`), should `done` be 1 or 0 in the target? Why?

**(b)** What would go wrong if we used `done = terminated or truncated`?

*Your answer:*

---
## Part 6 — Loss and SGD step (algorithm lines 11–12)

$$ l = \big(\delta_j - \hat Q(s_j, a_j; \theta)\big)^2 $$

### TODO 6.1 — `optimize_step`

1. Get $\hat Q(s_j, a_j; \theta)$ for the actions **actually taken**: `online_net(s).gather(1, a.unsqueeze(1)).squeeze(1)`.
2. Compute the target with `double_dqn_target` or `dqn_target`, depending on the `double` flag.
3. Mean-squared error, `zero_grad`, `backward`, clip gradients (`max_norm=10`), `step`.

The gradient flows **only into θ**. θ⁻ is updated later, by copying (line 13).

In [ ]:
def optimize_step(online_net, target_net, optimizer, buffer, batch_size, gamma, double):
    """One SGD step on a random mini-batch. Returns (loss, mean Q(s,a))."""
    s, a, r, s_next, done = buffer.sample(batch_size)          # line 9
    # YOUR CODE HERE
    raise NotImplementedError()
    return loss.item(), q_sa.mean().item()

In [ ]:
# ✔ Check 6.1
set_seed(1)
_on, _tg = QNetwork(4, 2).to(device), QNetwork(4, 2).to(device)
_tg.load_state_dict(_on.state_dict())
_opt = torch.optim.Adam(_on.parameters(), lr=1e-3)
_buf = ReplayBuffer(500)
for _ in range(200):
    _buf.push(np.random.randn(4).astype(np.float32), random.randrange(2), 1.0,
              np.random.randn(4).astype(np.float32), float(random.random() < 0.1))
_before_on = [p.detach().clone() for p in _on.parameters()]
_before_tg = [p.detach().clone() for p in _tg.parameters()]
_loss, _ = optimize_step(_on, _tg, _opt, _buf, 32, 0.99, double=True)

assert np.isfinite(_loss), "The loss must be a finite number."
assert any(not torch.equal(b, p) for b, p in zip(_before_on, _on.parameters())), "θ was not updated."
assert all(torch.equal(b, p) for b, p in zip(_before_tg, _tg.parameters())), "θ⁻ must NOT change in an SGD step."
assert all(p.grad is None for p in _tg.parameters()), "No gradient may reach θ⁻."
print(f"✅ Part 6 OK — loss = {_loss:.4f}")

---
## Part 7 — The full training loop (algorithm lines 1–13)

Most of the loop is provided; read it side by side with the algorithm box. You complete the two pieces that matter most.

### TODO 7.1 — Store the experience (line 6)
Push the transition into the buffer. Use **`terminated`**, not `truncated`, as the `done` flag (remember Question 5).

### TODO 7.2 — Warm-up gate, learning, target sync (lines 8–13)
* Learn only if `len(buffer) > min_replay` **and** `len(buffer) > batch_size` (line 8).
* Call `optimize_step` and append the loss to `history["losses"]`.
* Count gradient updates in `n_updates`; every `C` updates, copy θ into θ⁻ with `target_net.load_state_dict(online_net.state_dict())` (line 13).

The code also tracks the **average predicted value** $\frac{1}{N}\sum_i \max_a \hat Q(s_i, a; \theta)$ on a fixed set of "probe" states. We use it in Part 8 to watch overestimation happen.

In [ ]:
CONFIG = dict(
    env_id="CartPole-v1",
    gamma=0.99,             # discount rate γ
    lr=5e-4,                # learning rate α
    batch_size=64,          # b
    buffer_capacity=50_000, # N
    min_replay=1_000,       # m  (m ≥ b)
    C=500,                  # target sync interval, in gradient updates
    total_steps=50_000,     # K
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_steps=10_000,
    hidden=128,
    log_every=2_500,
)


def collect_probe_states(env_id, n=500, seed=123):
    """Fixed states (from a random policy) used to track the average predicted value."""
    env = gym.make(env_id)
    obs, _ = env.reset(seed=seed)
    states = []
    while len(states) < n:
        states.append(obs)
        obs, _, term, trunc, _ = env.step(env.action_space.sample())
        if term or trunc:
            obs, _ = env.reset()
    env.close()
    return torch.as_tensor(np.array(states), dtype=torch.float32, device=device)


def train(double: bool, seed: int = 0, cfg: dict = CONFIG, verbose: bool = True, **overrides):
    cfg = {**cfg, **overrides}
    set_seed(seed)
    env = gym.make(cfg["env_id"])
    obs_dim, n_actions = env.observation_space.shape[0], env.action_space.n

    # Initialize θ and θ⁻ = θ, and the replay buffer D
    online_net = QNetwork(obs_dim, n_actions, cfg["hidden"]).to(device)
    target_net = QNetwork(obs_dim, n_actions, cfg["hidden"]).to(device)
    target_net.load_state_dict(online_net.state_dict())
    optimizer = torch.optim.Adam(online_net.parameters(), lr=cfg["lr"])
    buffer = ReplayBuffer(cfg["buffer_capacity"])
    probe = collect_probe_states(cfg["env_id"])

    history = dict(returns=[], return_steps=[], q_probe=[], q_steps=[], losses=[])
    state, _ = env.reset(seed=seed)
    ep_return, n_updates = 0.0, 0

    for step in range(1, cfg["total_steps"] + 1):                       # line 1
        eps = linear_epsilon(step, cfg["eps_start"], cfg["eps_end"], cfg["eps_decay_steps"])  # line 5
        action = select_action(state, online_net, eps, n_actions)       # line 2
        next_state, reward, terminated, truncated, _ = env.step(action) # line 3

        # TODO 7.1 — store the experience (line 6)
        # YOUR CODE HERE
        raise NotImplementedError()

        state = next_state                                              # line 7
        ep_return += reward
        if terminated or truncated:
            history["returns"].append(ep_return)
            history["return_steps"].append(step)
            state, _ = env.reset()
            ep_return = 0.0

        # TODO 7.2 — warm-up gate, learning step, target sync (lines 8–13)
        # YOUR CODE HERE
        raise NotImplementedError()

        if step % cfg["log_every"] == 0:
            with torch.no_grad():
                q_mean = online_net(probe).max(dim=1).values.mean().item()
            history["q_probe"].append(q_mean)
            history["q_steps"].append(step)
            if verbose:
                recent = np.mean(history["returns"][-10:]) if history["returns"] else float("nan")
                print(f"{'Double' if double else 'Standard'} DQN | step {step:6d} | ε={eps:.2f} "
                      f"| return(last 10)={recent:6.1f} | avg max Q(probe)={q_mean:6.1f} | updates={n_updates}")
    env.close()
    history["n_updates"] = n_updates
    return online_net, history

In [ ]:
# ✔ Check 7 — a very short smoke test (a few seconds)
_net, _hist = train(double=True, seed=0, verbose=False, total_steps=3_000, min_replay=500, C=100, log_every=1_000)
assert _hist["n_updates"] == 3_000 - 500, \
    f"Expected 2500 updates (learning starts once len(D) > m), got {_hist['n_updates']}."
assert len(_hist["losses"]) == _hist["n_updates"], "Append one loss per gradient update."
assert len(_hist["returns"]) > 0, "Episodes should finish and be recorded."
print("✅ Part 7 OK — the training loop runs.")

---
## Part 8 — Experiment: standard DQN vs Double DQN

We train both agents with the **same seeds and hyperparameters**. The only difference is the TD target.

⏱️ Each run takes a few minutes on a laptop CPU. Start with `SEEDS = [0]`; add more seeds if time allows, since RL results are noisy.

In [ ]:
SEEDS = [0, 1]

results = {"Standard DQN": [], "Double DQN": []}
for seed in SEEDS:
    for name, double in [("Standard DQN", False), ("Double DQN", True)]:
        print(f"\n=== {name} | seed {seed} ===")
        net, hist = train(double=double, seed=seed)
        results[name].append((net, hist))

### Plotting helpers (provided)

In [ ]:
COLORS = {"Standard DQN": "#7A869A", "Double DQN": "#D9692A"}


def smooth(x, k=20):
    x = np.asarray(x, dtype=float)
    if len(x) < k:
        return x
    return np.convolve(x, np.ones(k) / k, mode="valid")


fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, runs in results.items():
    for i, (_, h) in enumerate(runs):
        y = smooth(h["returns"])
        x = np.array(h["return_steps"])[len(h["returns"]) - len(y):]
        axes[0].plot(x, y, color=COLORS[name], alpha=0.9, label=name if i == 0 else None)
        axes[1].plot(h["q_steps"], h["q_probe"], color=COLORS[name], marker="o", ms=3,
                     label=name if i == 0 else None)

axes[0].set(title="Episode return (smoothed)", xlabel="environment steps", ylabel="return")
axes[1].axhline(1 / (1 - CONFIG["gamma"]), color="#14213D", ls="--", lw=1.5,
                label="upper bound 1/(1−γ) = 100")
axes[1].set(title="Average predicted value  max_a Q(s,a;θ)  on probe states",
            xlabel="environment steps", ylabel="Q estimate")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

### TODO 8.1 — Measure overestimation directly

For a trained network, run **10 greedy episodes** (ε = 0) and, at the **first state** $s_0$ of each episode, compare:

* the **predicted** value $\max_a \hat Q(s_0, a; \theta)$, and
* the **actual** discounted return $G_0 = \sum_{t} \gamma^t r_t$ obtained from that state.

Return the mean predicted value, the mean $G_0$ and the mean undiscounted return. A positive gap $\hat Q - G_0$ means the network is **overestimating**.

In [ ]:
def evaluate(net, n_episodes=10, gamma=CONFIG["gamma"], seed=10_000):
    """Greedy evaluation. Returns (mean predicted Q(s0), mean discounted G0, mean return)."""
    env = gym.make(CONFIG["env_id"])
    preds, g0s, rets = [], [], []
    # YOUR CODE HERE
    raise NotImplementedError()
    env.close()
    return float(np.mean(preds)), float(np.mean(g0s)), float(np.mean(rets))

In [ ]:
print(f"{'method':<14}{'seed':>5}{'pred Q(s0)':>13}{'true G0':>10}{'gap':>9}{'return':>9}")
for name, runs in results.items():
    for seed, (net, _) in zip(SEEDS, runs):
        q_hat, g0, ret = evaluate(net)
        print(f"{name:<14}{seed:>5}{q_hat:>13.1f}{g0:>10.1f}{q_hat - g0:>+9.1f}{ret:>9.1f}")

### ✍️ Questions 8

**(a)** Look at the right-hand plot. Did standard DQN's average predicted value go **above 100**, the theoretical maximum? What about Double DQN?

**(b)** Compare the `gap` column for both methods. Relate what you see to the *winner's curse* from the slides.

**(c)** Did Double DQN learn faster, slower or about the same as standard DQN in terms of return? Why might the return curves look similar even when the Q-values differ?

*Your answers:*

---
## Part 9 — (Optional) Experiment: a larger target-sync interval C

The reference suggests that for Double DQN it can help to increase **C** to about **2×–3×** the standard DQN setting. A θ⁻ that differs more from θ is a more *independent judge*. If it differs too much, though, the targets come from a **stale** network.

### TODO 9.1
Train Double DQN with `C = 500` (already done in Part 8) and with `C = 1250` (2.5×) on the same seed, then plot both return curves and probe-Q curves.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

### ✍️ Questions 9

**(a)** What changed when C was larger: the stability of the return curve, the speed of learning, the Q-values?

**(b)** Thought experiment: what does Double DQN become if **C = 1**, i.e. θ⁻ is overwritten by θ after every update?

*Your answers:*

---
## Part 10 — Wrap-up

### ✍️ Final reflection

1. Point to the exact line(s) of **your** code where θ selects and θ⁻ evaluates.
2. Why is the target computed inside `torch.no_grad()`? What would change if it were not?
3. Name everything in your implementation that is **identical** to standard DQN.

*Your answers:*

### Submission
Submit this notebook with all cells executed, all checks showing ✅, both plots visible, and every ✍️ question answered.